# 04 · Multimodal: BLIP VQA on MS-COCO, CLIP, Whisper, and the LangChain chain
Buổi 16–17.
* **VQA**: `Salesforce/blip-vqa-base` on 1,000 VQAv2 validation questions (MS-COCO images), standard VQA accuracy. Target > 70%.
* **CLIP**: zero-shot mapping of product photos to the CVAE's 10 categories (checked on FashionMNIST).
* **Whisper**: speech-to-text feeding ABSA.
* **LangChain**: the exact `/analyze` chain the API runs (ingest → ABSA | long context | vision → fuse → generate). About 15 minutes on a T4.

In [ ]:
#@title Setup — run once per session (Runtime > Change runtime type > T4 GPU)
REPO_URL = "https://github.com/YoungsterDumb/reviewlens"  # @param {type:"string"}
HF_NAMESPACE = "DatDuc"  # @param {type:"string"}  your Hugging Face username
import os, sys, subprocess
if not os.path.exists("/content/reviewlens"):
    subprocess.run(["git", "clone", "-q", "--depth", "1", REPO_URL, "/content/reviewlens"], check=True)
else:
    subprocess.run(["git", "-C", "/content/reviewlens", "pull", "-q"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "/content/reviewlens/training/requirements.txt"], check=True)
sys.path.insert(0, "/content/reviewlens/training")
import torch
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none — switch the runtime to T4 GPU")

In [ ]:
from huggingface_hub import notebook_login
notebook_login()  # paste a token with WRITE access (huggingface.co/settings/tokens)
METRICS_REPO = f"{HF_NAMESPACE}/reviewlens-metrics"

In [ ]:
from reviewlens_train.evaluate import evaluate_vqa
vqa = evaluate_vqa(n=1000)
vqa

In [ ]:
from pathlib import Path
from reviewlens_train.data import load_fashion_mnist
from reviewlens_train.evaluate import evaluate_clip_categories
x, y = load_fashion_mnist(Path("/content/data/fashion"))
clip = evaluate_clip_categories(x, y, n=500)
clip

In [ ]:
from reviewlens_train.hub import update_metrics
update_metrics(METRICS_REPO, {**vqa, **clip})

## Whisper → ABSA, through the same LangChain chain as the API

In [ ]:
from datasets import load_dataset
from app.config import Settings
from app.registry import Registry
from app.pipeline.chain import build_analyze_chain, run_analyze
from PIL import Image
import numpy as np
registry = Registry(Settings(hf_namespace=HF_NAMESPACE, cvae_dir="/content/reviewlens/backend/assets/cvae"))
chain = build_analyze_chain(registry)
speech = load_dataset("hf-internal-testing/librispeech_asr_dummy", "clean", split="validation")[0]["audio"]
import librosa
audio = librosa.resample(np.asarray(speech["array"], dtype="float32"), orig_sr=speech["sampling_rate"], target_sr=16000)
photo = Image.fromarray((x[0] * 255).astype("uint8")).convert("RGB").resize((224, 224))
out = run_analyze(chain, text="The screen is great but the battery is terrible.", audio=audio, image=photo, lang="auto")
print("transcript:", out["transcript"])
print([(a["term"], a["sentiment"]) for a in out["aspects"]], out["priority"])
print(out["image_result"]["category"], out["image_result"]["answers"])
print(out["_trace"])